# Laboratório 10 — O Pipeline Definitivo

**RAG + QLoRA + FlashAttention-2 + KV Cache na GPU**

Disciplina: Tópicos em IA — Instituto de Ensino Superior iCEV

Aluno: Vinicius Henrique Albino Andrade

---

## 🚀 COMO RODAR (LEIA ANTES)

> **Procedimento padrão:**
> 1. **`Ambiente de execução`** → **`Alterar o tipo de ambiente`** → escolha **GPU T4** → Salvar
> 2. **`Ambiente de execução`** → **`Executar tudo`** (`Ctrl+F9`)
> 3. Aguarde ~15-20 min. **A célula do "Passo 3 (BASELINE sem cache)" demora 5-15 min** — é o ponto pedagógico do lab. Não cancele.
>
> **Se aparecer erro `is_flash_attn_4_available`** (incompatibilidade de versão do transformers):
> - Execute a primeira célula (pip install) → **`Ambiente de execução`** → **`Reiniciar sessão`** → **`Executar tudo`**

---

## Contexto
A HealthTech (Lab 09) colocou em produção um RAG médico que recupera ~5 capítulos inteiros de manuais (≈15k tokens) e injeta num LLM fine-tunado. O servidor estourou a VRAM por causa do custo O(n²) do Self-Attention.

Missão: orquestrar **QLoRA 4-bit** (Unidade II) + **KV Cache** + **FlashAttention-2** (Unidade I) para evitar OOM e ainda reduzir latência.

## Requisito de runtime
- **GPU NVIDIA com CUDA** (obrigatório por causa do `bitsandbytes`).
- FlashAttention-2 oficialmente exige **Ampere (sm_80) ou superior** (L4, A100, A10, H100). Se cair em T4 (Turing sm_75), o notebook faz **fallback automático para `sdpa`** — PyTorch Scaled Dot Product Attention — que também usa kernels memory-efficient e preserva o ganho de memória da fase de prompting, embora não seja exatamente o FA-2.

## Setup — instalação de dependências

A célula abaixo verifica se as versões corretas já estão instaladas. Se precisar instalar, ela faz o pip install **e reinicia a sessão automaticamente**. Caso isso aconteça, é só clicar em `Executar tudo` novamente — da segunda vez ela passa direto.

In [ ]:
# ============================================================
# CÉLULA 1 — Setup do ambiente (versões compatíveis com Colab atual)
# ============================================================
# Notas de compatibilidade:
#  - transformers 5.x tem bug import 'is_flash_attn_4_available' -> pin 4.46.3
#  - bitsandbytes <=0.44 depende de triton.ops (removido no triton 3.x do Colab atual)
#    e não tem binário para CUDA 12.8 -> precisa >=0.45
import os, subprocess, sys

TARGET_TRANSFORMERS = "4.46.3"
TARGET_BNB = "0.45.3"

def get_version(pkg):
    try:
        out = subprocess.check_output([sys.executable, "-m", "pip", "show", pkg], text=True)
        for line in out.splitlines():
            if line.startswith("Version:"):
                return line.split(":", 1)[1].strip()
    except subprocess.CalledProcessError:
        return None
    return None

cur_tf = get_version("transformers")
cur_bnb = get_version("bitsandbytes")
needs_install = (cur_tf != TARGET_TRANSFORMERS) or (cur_bnb != TARGET_BNB)

if needs_install:
    print(f"transformers atual: {cur_tf} -> {TARGET_TRANSFORMERS}")
    print(f"bitsandbytes atual: {cur_bnb} -> {TARGET_BNB}")
    print("Instalando versoes compativeis com Colab (CUDA 12.x + triton 3.x)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U",
        f"transformers=={TARGET_TRANSFORMERS}",
        "accelerate==1.1.1",
        f"bitsandbytes=={TARGET_BNB}",
        "sentencepiece", "matplotlib"])
    # flash-attn opcional (falha em T4 e' esperada)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "flash-attn>=2.5.0", "--no-build-isolation"], check=False)
    print("\nInstalacao concluida.")

    # Mata sessao se ja tinha transformers/bitsandbytes carregados em memoria
    if "transformers" in sys.modules or "bitsandbytes" in sys.modules:
        print("\n>>> Reiniciando sessao automaticamente para carregar as novas versoes...")
        print(">>> Apos o restart, clique em 'Ambiente de execucao' -> 'Executar tudo'.")
        os.kill(os.getpid(), 9)
else:
    print(f"transformers={cur_tf}, bitsandbytes={cur_bnb} -> tudo certo, seguindo.")

In [ ]:
import os, gc, time, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "GPU CUDA é obrigatória para este laboratório."

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
SUPPORTS_FA2 = CAPABILITY[0] >= 8  # Ampere ou superior

print(f"GPU detectada : {GPU_NAME}")
print(f"Compute capab.: {CAPABILITY}  (sm_{CAPABILITY[0]}{CAPABILITY[1]})")
print(f"Suporta FA-2  : {SUPPORTS_FA2}")
print(f"VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB")

---

## Passo 1 — Ingestão Eficiente (QLoRA 4-bit via bitsandbytes)

Carregar o LLM base em FP16 ocuparia ~2.2 GB só para o TinyLlama-1.1B. Em modelos maiores isso já estoura VRAM. **QLoRA quantiza os pesos do modelo congelado para 4 bits (NF4)** e mantém o cálculo em FP16, reduzindo a pegada de memória em ~4× sem perda significativa de qualidade.

Configuração usada:
- `load_in_4bit=True`
- `bnb_4bit_quant_type="nf4"` (4-bit NormalFloat — distribuição otimizada para pesos)
- `bnb_4bit_compute_dtype=torch.float16`
- `bnb_4bit_use_double_quant=True` (quantiza também as constantes de quantização → economia extra)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
vram_before = torch.cuda.memory_allocated() / 1024**2

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

vram_after = torch.cuda.memory_allocated() / 1024**2
vram_model_only = vram_after - vram_before

print(f"VRAM antes do load        : {vram_before:.2f} MB")
print(f"VRAM após carregar modelo : {vram_after:.2f} MB")
print(f"Pegada do modelo quantizado: {vram_model_only:.2f} MB")
print(f"\n[MÉTRICA — PASSO 1] Modelo {MODEL_ID} em 4-bit NF4 ocupa ~{vram_model_only:.0f} MB de VRAM.")

---

## Passo 2 — Simulando o RAG Massivo (Lab 09)

No Lab 09 construímos um RAG com HNSW + HyDE + Cross-Encoder para manuais médicos. Aqui simulamos o que esse pipeline entregaria em produção: **5 capítulos inteiros recuperados** totalizando ~12-15 mil tokens.

O texto fictício abaixo foi **gerado por IA e revisado manualmente** (uso permitido pela seção 4.2 do contrato pedagógico — geração de texto base fictício). Mantém jargão clínico coerente com o caso da HealthTech.

In [ ]:
MEDICAL_CHAPTERS = [
    # Cap. 1 - Cefaleias primárias
    """CAPÍTULO 1 — CEFALEIAS PRIMÁRIAS E DIAGNÓSTICO DIFERENCIAL.
A cefaleia pulsátil unilateral acompanhada de fotofobia, fonofobia, náusea e piora ao esforço físico configura o quadro clássico de migrânea sem aura, conforme classificação ICHD-3. O diagnóstico é eminentemente clínico, mas exames de imagem (RM de crânio) são indicados quando há sinais de alarme: início abrupto em thunderclap, idade superior a 50 anos no primeiro episódio, mudança de padrão habitual, déficit neurológico focal persistente, febre associada, papiledema ou cefaleia desencadeada por manobra de Valsalva.
O tratamento abortivo de primeira linha em adultos inclui triptanos (sumatriptano 50-100mg VO, zolmitriptano 2.5-5mg VO) associados a antieméticos como metoclopramida 10mg. AINEs como naproxeno 550mg ou cetorolaco 30mg IM são alternativas para pacientes com contraindicação a triptanos. A profilaxia é considerada quando há mais de 4 crises mensais ou impacto funcional significativo, com opções incluindo propranolol 80-160mg/dia, topiramato 50-100mg/dia, amitriptilina 25-75mg/dia e, mais recentemente, anticorpos monoclonais anti-CGRP (erenumabe, galcanezumabe, fremanezumabe).
A cefaleia tensional, em contraste, apresenta-se como dor bilateral, em aperto, intensidade leve a moderada, sem fotofobia significativa e sem piora com atividade. Cefaleia em salvas (cluster) caracteriza-se por dor periorbital extrema, unilateral, com manifestações autonômicas ipsilaterais (lacrimejamento, rinorreia, ptose, miose), em surtos de 15-180 minutos múltiplas vezes ao dia durante semanas. Tratamento agudo: oxigênio 100% a 12L/min por máscara não-reinalante e sumatriptano subcutâneo 6mg. Profilaxia: verapamil 240-480mg/dia.
""",
    # Cap. 2 - Dor torácica
    """CAPÍTULO 2 — AVALIAÇÃO DA DOR TORÁCICA NA EMERGÊNCIA.
A síndrome coronariana aguda (SCA) deve ser a primeira hipótese a ser descartada em todo paciente adulto com dor torácica aguda. O eletrocardiograma de 12 derivações deve ser realizado em até 10 minutos da chegada. Achados como supradesnivelamento do segmento ST (STEMI), infradesnivelamento, inversão de onda T ou bloqueio de ramo esquerdo novo são altamente sugestivos. A troponina cardíaca de alta sensibilidade deve ser dosada na admissão e repetida em 1-3 horas, conforme protocolos rápidos da ESC.
O escore HEART (History, ECG, Age, Risk factors, Troponin) estratifica o risco em baixo (0-3), intermediário (4-6) e alto (>=7), guiando a conduta. Pacientes de baixo risco podem ter alta com follow-up ambulatorial em 72h; intermediário/alto risco necessitam internação, dupla antiagregação plaquetária (AAS 300mg + clopidogrel 600mg ou ticagrelor 180mg), anticoagulação (enoxaparina 1mg/kg SC 12/12h ou heparina não-fracionada) e estratificação invasiva.
Diagnósticos diferenciais críticos incluem dissecção aórtica (dor migratória, assimetria de pulsos, sopro de regurgitação aórtica nova — angiotomografia urgente), tromboembolismo pulmonar (dispneia súbita, taquicardia, hipoxemia, D-dímero elevado — angio-TC de tórax), pneumotórax hipertensivo (timpanismo, abolição do murmúrio vesicular, desvio de traqueia — drenagem imediata sem aguardar imagem), pericardite (dor pleurítica que melhora ao se inclinar para frente, atrito pericárdico, supra-ST difuso côncavo) e tamponamento cardíaco (tríade de Beck: hipotensão, turgência jugular, abafamento de bulhas).
""",
    # Cap. 3 - Sepse
    """CAPÍTULO 3 — RECONHECIMENTO E MANEJO DA SEPSE.
Sepse é definida pelo consenso Sepsis-3 como disfunção orgânica ameaçadora à vida causada por resposta desregulada do hospedeiro à infecção. Operacionalmente, identifica-se por aumento >=2 pontos no escore SOFA na presença de infecção suspeita ou confirmada. O qSOFA (frequência respiratória >=22, alteração de consciência com Glasgow <15, pressão sistólica <=100) é uma ferramenta de triagem rápida — qSOFA >=2 alerta para risco aumentado de mortalidade.
Choque séptico é o subgrupo de sepse que apresenta anormalidades circulatórias, celulares e metabólicas profundas o suficiente para aumentar substancialmente a mortalidade, identificado clinicamente por necessidade de vasopressor para manter PAM >=65 mmHg e lactato sérico >2 mmol/L apesar de ressuscitação volêmica adequada.
O bundle de 1 hora da Surviving Sepsis Campaign preconiza: (1) dosar lactato e repetir se >2; (2) coletar hemoculturas antes dos antibióticos quando possível, sem atrasar a antibioticoterapia; (3) administrar antibiótico de amplo espectro endovenoso em até 1 hora; (4) iniciar cristaloide 30 mL/kg IV em 3 horas se hipotensão ou lactato >=4; (5) iniciar vasopressor (noradrenalina como primeira linha) se hipotensão persistente após reposição, alvo PAM >=65 mmHg.
A escolha empírica de antimicrobiano deve cobrir gram-positivos, gram-negativos e considerar fatores como foco presumido, exposição prévia, residência em ILPI e padrão local de resistência. Esquemas usuais: piperacilina-tazobactam 4.5g IV 6/6h +/- vancomicina 25-30 mg/kg dose de ataque; em pacientes com risco de pseudomonas multirresistente, meropenem 1g IV 8/8h. Descalonamento orientado por culturas em 48-72h é essencial para stewardship antimicrobiano.
""",
    # Cap. 4 - AVE
    """CAPÍTULO 4 — ACIDENTE VASCULAR ENCEFÁLICO ISQUÊMICO AGUDO.
O AVE isquêmico é uma emergência tempo-dependente: "time is brain" — estima-se a perda de 1.9 milhão de neurônios por minuto de isquemia. A escala NIHSS quantifica o déficit neurológico (0-42 pontos) e orienta a decisão terapêutica. A escala FAST (Face, Arm, Speech, Time) é a ferramenta de triagem pré-hospitalar mais difundida.
A tomografia de crânio sem contraste é o exame inicial obrigatório, com finalidade primária de excluir hemorragia. ASPECTS <=7 sugere área isquêmica extensa e relativiza o benefício da trombólise. Em centros com disponibilidade, angio-TC de cabeça e pescoço e TC de perfusão guiam a decisão sobre trombectomia mecânica em pacientes com oclusão de grande vaso.
Trombólise endovenosa com alteplase 0.9 mg/kg (10% em bolus, restante em 1h, máximo 90mg) está indicada em até 4.5 horas do ictus para pacientes elegíveis. Tenecteplase 0.25 mg/kg em bolus único vem demonstrando não-inferioridade. Contraindicações absolutas incluem hemorragia intracraniana prévia, AVE isquêmico nos últimos 3 meses, sangramento ativo, plaquetas <100.000, INR >1.7, uso de DOACs nas últimas 48h sem reversão.
Trombectomia mecânica está indicada em oclusão de grande vaso da circulação anterior em janela de até 6 horas (ou até 24h em pacientes selecionados pelo critério DAWN/DEFUSE-3 com mismatch core-penumbra). O manejo da pressão arterial pré-trombólise visa <185/110; pós-trombólise <180/105 nas primeiras 24h. Hipertensão permissiva em candidatos a trombectomia mantém PA <220/110.
""",
    # Cap. 5 - Insuficiência respiratória
    """CAPÍTULO 5 — INSUFICIÊNCIA RESPIRATÓRIA AGUDA E VENTILAÇÃO MECÂNICA.
Insuficiência respiratória classifica-se em tipo I (hipoxêmica, PaO2 <60 mmHg em ar ambiente) e tipo II (hipercápnica, PaCO2 >45 mmHg com acidemia). A oximetria de pulso é triagem inicial, mas a gasometria arterial é o padrão-ouro diagnóstico, fornecendo PaO2, PaCO2, pH, bicarbonato e lactato. A relação PaO2/FiO2 (P/F) estratifica a gravidade da SDRA segundo Berlin: leve 200-300, moderada 100-200, grave <100.
Suporte ventilatório não-invasivo (VNI) com BiPAP é primeira escolha em DPOC exacerbada com acidose respiratória (pH 7.25-7.35) e em edema agudo de pulmão cardiogênico. CNAF (cateter nasal de alto fluxo) é alternativa em insuficiência respiratória hipoxêmica não-hipercápnica, especialmente em pacientes imunocompetentes com pneumonia.
Indicações de intubação orotraqueal incluem rebaixamento do nível de consciência com Glasgow <=8, instabilidade hemodinâmica grave, falência de VNI/CNAF, parada respiratória iminente e necessidade de proteção de via aérea. A sequência rápida de intubação utiliza pré-oxigenação 3-5 minutos com FiO2 100%, indutor (etomidato 0.3 mg/kg, propofol 1.5-2 mg/kg, cetamina 1-2 mg/kg) e bloqueador neuromuscular (succinilcolina 1.5 mg/kg ou rocurônio 1.2 mg/kg).
Em SDRA, a ventilação protetora preconiza volume corrente 4-8 mL/kg de peso predito, pressão de platô <=30 cmH2O, driving pressure <=15, PEEP titulada conforme tabela ARDSnet ou estratégia de PEEP alta em casos selecionados, hipercapnia permissiva (pH >=7.20). Posição prona por >=16 horas/dia reduz mortalidade em SDRA grave (P/F <150). Bloqueio neuromuscular contínuo nas primeiras 48h pode ser considerado em casos refratários. ECMO veno-venosa é resgate em casos selecionados.
""",
]

# Replicamos o conteúdo algumas vezes para atingir ~12-15k tokens (simulação do funil largo do RAG)
rag_context = "\n\n".join(MEDICAL_CHAPTERS * 3)

print(f"Caracteres no contexto simulado: {len(rag_context):,}")
print(f"Capítulos lógicos: 5 (replicados 3x para simular volume do RAG)")

In [ ]:
PROMPT_INSTRUCTION = (
    "\n\n[INSTRUÇÃO CLÍNICA]\n"
    "Com base nos capítulos médicos acima recuperados pelo sistema RAG, "
    "redija um resumo clínico estruturado de aproximadamente 500 palavras "
    "para um plantonista de emergência, destacando os sinais de alarme, "
    "critérios diagnósticos e condutas de primeira linha.\n\n[RESUMO]:\n"
)

full_prompt = rag_context + PROMPT_INSTRUCTION

encoded = tokenizer(full_prompt, return_tensors="pt").to(model.device)
input_ids = encoded.input_ids
n_input_tokens = input_ids.shape[1]

print(f"[MÉTRICA — PASSO 2] Contexto tokenizado: {n_input_tokens:,} tokens")
print(f"Shape de input_ids: {tuple(input_ids.shape)}")

---

## Passo 3 — O Gargalo de Geração (SEM KV Cache)

A geração autorregressiva produz um token de cada vez. Sem KV Cache, a cada novo token o modelo **recalcula Q, K, V de todos os tokens anteriores** — incluindo os 12k tokens do contexto RAG. Isso transforma a geração de N tokens em uma operação O(N · L²) onde L é o tamanho do prompt.

**Pegadinha exigida pelo lab**: `model.config.use_cache = False`.

Vamos medir:
- ⏱️ Tempo total para gerar 100 tokens
- 🧠 Pico de VRAM (`torch.cuda.max_memory_allocated`)
- 📈 Tokens/segundo (throughput)

> ⚠️ Esta célula pode levar **vários minutos** — é justamente o ponto: demonstrar que sem cache a inferência é proibitiva em produção.

In [ ]:
def benchmark_generation(model, input_ids, max_new_tokens=100, use_cache=True, label=""):
    """Gera max_new_tokens e retorna métricas de tempo e VRAM."""
    model.config.use_cache = use_cache
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    vram_start = torch.cuda.memory_allocated() / 1024**2

    # warmup + sincroniza GPU antes de medir
    torch.cuda.synchronize()
    t0 = time.time()

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy — benchmark determinístico
            use_cache=use_cache,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    elapsed = time.time() - t0

    vram_peak = torch.cuda.max_memory_allocated() / 1024**2
    n_generated = output.shape[1] - input_ids.shape[1]
    tokens_per_sec = n_generated / elapsed if elapsed > 0 else 0

    print(f"\n=== {label} ===")
    print(f"  use_cache         : {use_cache}")
    print(f"  Tokens gerados    : {n_generated}")
    print(f"  Tempo total       : {elapsed:.2f} s")
    print(f"  Throughput        : {tokens_per_sec:.2f} tok/s")
    print(f"  VRAM no início    : {vram_start:.2f} MB")
    print(f"  VRAM pico         : {vram_peak:.2f} MB")
    print(f"  Delta VRAM (pico) : {vram_peak - vram_start:.2f} MB")

    return {
        "label": label,
        "use_cache": use_cache,
        "elapsed_s": elapsed,
        "tokens_per_sec": tokens_per_sec,
        "vram_peak_mb": vram_peak,
        "vram_delta_mb": vram_peak - vram_start,
        "n_generated": n_generated,
    }

In [ ]:
# Baseline: SEM KV Cache (a pegadinha do lab)
metrics_baseline = benchmark_generation(
    model,
    input_ids,
    max_new_tokens=100,
    use_cache=False,
    label="BASELINE — QLoRA 4-bit, attention=eager, use_cache=False",
)

---

## Passo 4 — A Engenharia de Otimização

Aplicamos duas otimizações ortogonais:

1. **KV Cache** (otimização de software): em vez de recalcular Q, K, V de todo o contexto a cada novo token, guardamos os tensores K e V já computados. A geração passa de O(L²·N) para O(L² + L·N) — ganho gigante quando L (contexto RAG) >> N (tokens gerados).

2. **FlashAttention-2** (otimização de hardware): o cálculo do softmax(QKᵀ/√d) tradicional materializa a matriz N×N inteira na **HBM** (memória global da GPU). FA-2 reformula o algoritmo em **tiling** que cabe na **SRAM** on-chip, evitando o write-back da matriz de atenção. Ganho de memória: O(N) em vez de O(N²) na fase de prompting.

> **Fallback**: se a GPU não for Ampere+ (T4, por ex.), usamos `attn_implementation="sdpa"` — o PyTorch nativo também faz attention memory-efficient, preservando o ganho conceitual.

In [ ]:
# Libera o modelo eager antes de recarregar com FlashAttention-2
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Decide o backend de atenção conforme suporte da GPU
if SUPPORTS_FA2:
    try:
        import flash_attn  # noqa: F401
        attn_impl = "flash_attention_2"
        backend_label = "FlashAttention-2"
    except ImportError:
        attn_impl = "sdpa"
        backend_label = "SDPA (flash-attn não instalado)"
else:
    attn_impl = "sdpa"
    backend_label = "SDPA (GPU pré-Ampere, FA-2 indisponível)"

print(f"Backend de atenção escolhido: {attn_impl}  ({backend_label})")

model_opt = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_impl,
)
model_opt.eval()

vram_after_opt = torch.cuda.memory_allocated() / 1024**2
print(f"VRAM após carregar modelo otimizado: {vram_after_opt:.2f} MB")

In [ ]:
# Re-tokeniza no novo device (o modelo pode ter sido movido)
encoded_opt = tokenizer(full_prompt, return_tensors="pt").to(model_opt.device)
input_ids_opt = encoded_opt.input_ids

# Otimizado: COM KV Cache + FA-2/SDPA
metrics_optimized = benchmark_generation(
    model_opt,
    input_ids_opt,
    max_new_tokens=100,
    use_cache=True,
    label=f"OTIMIZADO — QLoRA 4-bit, attention={attn_impl}, use_cache=True",
)

---

## Comparação Final — Tabela de Benchmark e Visualização

Consolidação das métricas dos Passos 3 e 4 para colar no README.md.

In [ ]:
def fmt_speedup(baseline, optimized):
    return f"{baseline / optimized:.2f}x mais rápido" if optimized > 0 else "n/a"

def fmt_memreduction(baseline, optimized):
    if baseline <= 0:
        return "n/a"
    pct = 100 * (1 - optimized / baseline)
    return f"{pct:.1f}% menos VRAM"

print("=" * 78)
print(f"  RELATÓRIO DE BENCHMARK — Lab 10")
print(f"  GPU: {GPU_NAME}  |  Modelo: {MODEL_ID}")
print(f"  Contexto RAG: {n_input_tokens:,} tokens  |  Tokens gerados: 100")
print("=" * 78)
print()
print(f"{'Métrica':<28}{'Baseline (no cache)':>22}{'Otimizado':>20}{'Δ':>15}")
print("-" * 78)
print(f"{'Tempo total (s)':<28}{metrics_baseline['elapsed_s']:>22.2f}{metrics_optimized['elapsed_s']:>20.2f}{fmt_speedup(metrics_baseline['elapsed_s'], metrics_optimized['elapsed_s']):>15}")
print(f"{'Throughput (tok/s)':<28}{metrics_baseline['tokens_per_sec']:>22.2f}{metrics_optimized['tokens_per_sec']:>20.2f}{'—':>15}")
print(f"{'VRAM pico (MB)':<28}{metrics_baseline['vram_peak_mb']:>22.2f}{metrics_optimized['vram_peak_mb']:>20.2f}{fmt_memreduction(metrics_baseline['vram_peak_mb'], metrics_optimized['vram_peak_mb']):>15}")
print(f"{'Δ VRAM gerado (MB)':<28}{metrics_baseline['vram_delta_mb']:>22.2f}{metrics_optimized['vram_delta_mb']:>20.2f}{fmt_memreduction(metrics_baseline['vram_delta_mb'], metrics_optimized['vram_delta_mb']):>15}")
print("=" * 78)

# Persiste métricas em JSON para o README
report = {
    "gpu": GPU_NAME,
    "compute_capability": f"sm_{CAPABILITY[0]}{CAPABILITY[1]}",
    "model_id": MODEL_ID,
    "context_tokens": int(n_input_tokens),
    "generated_tokens": 100,
    "attn_backend_optimized": attn_impl,
    "vram_model_4bit_mb": round(vram_model_only, 2),
    "baseline": metrics_baseline,
    "optimized": metrics_optimized,
    "speedup_x": round(metrics_baseline["elapsed_s"] / metrics_optimized["elapsed_s"], 2) if metrics_optimized["elapsed_s"] > 0 else None,
    "vram_peak_reduction_pct": round(100 * (1 - metrics_optimized["vram_peak_mb"] / metrics_baseline["vram_peak_mb"]), 1) if metrics_baseline["vram_peak_mb"] > 0 else None,
}
with open("benchmark_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print("\nMétricas salvas em benchmark_report.json")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

labels = ["Baseline\n(no cache, eager)", f"Otimizado\n(KV cache + {attn_impl})"]
colors = ["#d62728", "#2ca02c"]

# Tempo
times = [metrics_baseline["elapsed_s"], metrics_optimized["elapsed_s"]]
axes[0].bar(labels, times, color=colors)
axes[0].set_title("Tempo de geração (100 tokens)")
axes[0].set_ylabel("segundos")
for i, v in enumerate(times):
    axes[0].text(i, v + max(times) * 0.02, f"{v:.2f}s", ha="center", fontweight="bold")

# VRAM pico
vrams = [metrics_baseline["vram_peak_mb"], metrics_optimized["vram_peak_mb"]]
axes[1].bar(labels, vrams, color=colors)
axes[1].set_title("Pico de VRAM durante geração")
axes[1].set_ylabel("MB")
for i, v in enumerate(vrams):
    axes[1].text(i, v + max(vrams) * 0.02, f"{v:.0f} MB", ha="center", fontweight="bold")

plt.suptitle(f"Lab 10 — Benchmark de Inferência ({GPU_NAME}, contexto {n_input_tokens:,} tokens)", fontweight="bold")
plt.tight_layout()
plt.savefig("benchmark_chart.png", dpi=120, bbox_inches="tight")
plt.show()
print("Gráfico salvo em benchmark_chart.png")